# Face Recognition with ArcFace

This notebook demonstrates a face recognition pipeline based on ArcFace embeddings.

Pipeline:

1. Load dataset
2. Detect faces
3. Extract ArcFace embeddings
4. Compare embeddings using cosine similarity
5. Save matched images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall onnxruntime onnxruntime-gpu -y
!pip install opencv-python matplotlib numpy scikit-learn onnxruntime-gpu insightface

In [ ]:
import cv2
import os
import torch
import numpy as np
import pickle
from tqdm import tqdm
from insightface.app import FaceAnalysis

In [ ]:
ROOT_FOLDER = "/content/drive/MyDrive/Face_Recognition/DOU_DAY_2026"
MY_IMAGES_FOLDER = "/content/drive/MyDrive/Face_Recognition/Me"

EMBEDDINGS_PATH = "/content/drive/MyDrive/Face_Recognition/face_embeddings_ArcFace.pkl"
NP_EMBEDDINGS = "/content/drive/MyDrive/Face_Recognition/embeddings_ArcFace.npy"

OUTPUT_DIR = "/content/drive/MyDrive/Face_Recognition/Me/Faces_Output"

SIMILARITY_THRESHOLD = 0.5

In [ ]:
import onnxruntime as ort

print("ONNX providers:", ort.get_available_providers())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("GPU not available")

print(os.path.exists(ROOT_FOLDER))
print(os.path.exists(MY_IMAGES_FOLDER))

In [ ]:
app = FaceAnalysis(
    providers=["CUDAExecutionProvider"]
)

app.prepare(ctx_id=0)

In [ ]:
for name, model in app.models.items():
    print(name)
    print(model.session.get_providers())

In [ ]:
def collect_images(folder):
    paths = []

    for root, _, files in os.walk(folder):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(root, file))

    return paths

In [ ]:
image_paths = collect_images(ROOT_FOLDER)
print(len(image_paths))

In [ ]:
my_images = [
    cv2.imread(path)
    for path in collect_images(MY_IMAGES_FOLDER)
]
my_faces_encodings = []

for img in my_images:
  faces = app.get(img)

  if len(faces) == 0:
      continue

  my_faces_encodings.append(faces[0].embedding)

In [ ]:
def load_embeddings(path):

    if not os.path.exists(path):
        return []

    with open(path, "rb") as f:
        return pickle.load(f)

In [ ]:
metadata = load_embeddings(EMBEDDINGS_PATH)
print(len(metadata))

In [ ]:
def save_embeddings(new_metadata, path):
  with open(path, "wb") as f:
    pickle.dump(new_metadata, f)

In [ ]:
selected_paths = image_paths[0:1]

pbar = tqdm(enumerate(selected_paths), total=len(selected_paths))

# Extract ArcFace embeddings from every detected face
for i, path in pbar:
    image = cv2.imread(path)
    if image is None:
        continue  # skip broken images

    faces = app.get(image)

    for face in faces:
      metadata.append({
        "embedding": face.embedding,
        "path": path,
        "location": face.bbox
      })

    save_embeddings(metadata, EMBEDDINGS_PATH)
    pbar.set_description(f"Scanned: {i + 1}")

print("Complete!")
print(len(metadata))

In [ ]:
all_embeddings_np = np.array([data["embedding"] for data in metadata])
np.save(NP_EMBEDDINGS, all_embeddings_np)

In [ ]:
def cosine_similarity(a, b):

    return np.dot(a, b) / (
        np.linalg.norm(a) *
        np.linalg.norm(b)
    )

In [ ]:
my_photos = []

# Compare my embeddings with dataset embeddings
for item in metadata:
  for img_enc in my_faces_encodings:
    sim = cosine_similarity(img_enc, item["embedding"])

    if sim > SIMILARITY_THRESHOLD:
        my_photos.append(item["path"])

In [ ]:
print("Found matches:")
print(f"My photos: {len(my_photos)}")

In [ ]:
if len(my_photos) > 0:
  os.makedirs(OUTPUT_DIR, exist_ok=True)

for photo in my_photos:
  filename = os.path.basename(photo)
  output_path = os.path.join(OUTPUT_DIR, filename)
  img = cv2.imread(photo)
  cv2.imwrite(output_path, img)

## Results

The notebook successfully:

- detected faces with InsightFace
- extracted ArcFace embeddings
- compared embeddings using cosine similarity
- saved matched images